In [ ]:
%pip install -r requirements.txt

In [ ]:
%pip install -e .

In [ ]:
%pip install src/llama_recipes/transformers_minimal/.

In [ ]:
!pip uninstall llama_recipes

In [ ]:
# Dataset pre_processing (UNCONDITIONED)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\data_preprocess.py" \
  --dataset_name lakhmidi_dataset \
  --dataset_folder "C:\Users\Michael\Downloads\RockDataset" \
  --output_folder "C:\Users\Michael\Desktop\MusicDatasets\Datasets\RockDataset" \
  --model_config "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config.json" \
  --train_test_split_file None \
  --train_ratio 0.9 \
  --ts_threshold None

In [ ]:
# Dataset pre_processing per (CONDITIONED)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\data_preprocess_conditioned.py" \
  --dataset_name commu_con_gen \
  --dataset_folder "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\dataset\commu_midi" \
  --output_folder "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\processed" \
  --model_config "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config.json" \
  --train_test_split_file "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\dataset\commu_meta.csv" \
  --ts_threshold None
  --train_ratio

In [ ]:
# Distillation from unconditioned checkpoint (without fine-tuned weights)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\recipes\distillation\distillation_uncon_lahk.py" \
    --teacher_model_config "src/llama_recipes/configs/model_config.json" \
    --teacher_model_checkpoint "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
    --student_config_file "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config_tiny.json" \
    --data_dir "C:\Users\Michael\Desktop\MusicDatasets\Datasets\Moonbeam_Distillation\processed_data" \
    --csv_file "C:\Users\Michael\Desktop\MusicDatasets\Datasets\Moonbeam_Distillation\processed_data\train_test_split.csv" \
    --output_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\distilled" \
    --num_epochs 5 \
    --batch_size 1 \
    --lr 2e-4

In [ ]:
# TRAINING DEL MODELLO DISTILLATO (TINY)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\recipes\distillation\distillation.py" \
    --model_config_file "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\student_100M_con.json" \
    --output_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\distilled_100M_model" \
    --teacher_model_checkpoint "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
    --teacher_model_config "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config_commu_con_gen.json" \
    --teacher_peft_weights "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\peft_model" \
    --tokenizer_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\tokenizer.model" \
    --dataset "commu_con_gen_dataset" \
    --dataset_config_file "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\processed\train_test_split.csv" \
    --additional_token_dict "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\processed\indexed_tokens_dict.json" \
    --num_epochs 3 \
    --batch_size_training 4 \
    --lr 1e-4 \
    --temperature 2.0 \
    --alpha 0.5

In [ ]:
# Generate music from a primer MIDI (UNCONDITIONED)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\recipes\inference\custom_music_generation\continue_unconditioned.py" \
  --midi_input_path "C:\Users\Michael\Desktop\Generazioni_Moonbeam\RockTest.mid" \
  --output_path "C:\Users\Michael\Desktop\Generazioni_Moonbeam\brano_continuato.mid" \
  --ckpt_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
  --tokenizer_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model" \
  --model_config_path "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config.json" \
  --max_gen_len 1024 \
  --prompt_len 512 \
  --temperature 1.0 \
  --finetuned_PEFT_weight_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\peft_model"

In [ ]:
# Merge model for Conditioned Music Generation
!python recipes/inference/custom_music_generation/merge_peft_model.py \
    --base_model_config_path "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config.json" \
    --base_model_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
    --peft_model_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\peft_model" \
    --output_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\merged_model"

In [ ]:
# Generate conditional music from a csv file with metadata (CONDITIONED) (# PARAMETRO AGGIUNTIVO FACOLTATIVO: --chords "Am G C F Dm E7 Am Am")
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\recipes\inference\custom_music_generation\generate_single_midi.py" \
    --ckpt_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
    --tokenizer_path tokenizer.model \
    --model_config_path "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config_commu_con_gen.json" \
    --additional_token_dict_path "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\processed\indexed_tokens_dict.json" \
    --chord_dict_path "C:\Users\Michael\Desktop\MusicDatasets\Datasets\ComMU\processed\chord_dictionary.json" \
    --finetuned_PEFT_weight_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\peft_model" \
    --output_path "C:\Users\Michael\Desktop\Generazioni_Moonbeam\fast_classical_5.mid" \
    --max_gen_len 500 \
    --min_gen_len 200 \
    --audio_key "cmajor" \
    --bpm 120 \
    --genre "cinematic" \
    --inst "acoustic_piano" \
    --track_role "main_melody" \
    --num_measures 16 \
    --temperature 1.3 \
    --top_p 0.85 \
    --pitch_range "mid_high" \
    --chords "C G/B Am Em/G F C/E Dm7 G7 C"\
    --number_generations 4

In [ ]:
# Generate Unconditioned music from a model (UNCONDITIONED)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\recipes\inference\custom_music_generation\generate_uncon_midi.py" \
    --ckpt_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
    --model_config_path "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config.json" \
    --tokenizer_path tokenizer.model \
    --finetuned_PEFT_weight_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\peft_ROCK\30-127.safetensors-20250820T235432Z-1-001\30-127.safetensors" \
    --output_path "C:\Users\Michael\Desktop\Generazioni_Moonbeam\RockTest.mid" \
    --data_dir "C:\Users\Michael\Desktop\MusicDatasets\Datasets\RockDataset\processed" \
    --number_generations 4 \
    --prompt_len 50 \
    --max_gen_len 512\
    --temperature 1.0 \
    --top_p 0.95 \

In [ ]:
# Generate different versions of a song (UNCONDITIONED)
!python "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\recipes\inference\custom_music_generation\generate_variations_uncon_midi.py" \
    --ckpt_dir "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\base_model\moonbeam_839M.pt" \
    --model_config_path "C:\Users\Michael\Desktop\Moonbeam-MIDI-Distillation\src\llama_recipes\configs\model_config.json" \
    --tokenizer_path "tokenizer.model" \
    --finetuned_PEFT_weight_path "C:\Users\Michael\Desktop\ModelliMusicGenerator\MOONBEAM\peft_ROCK\30-127.safetensors-20250820T235432Z-1-001\30-127.safetensors" \
    --output_path "C:\Users\Michael\Desktop\Generazioni_Moonbeam\RockVariations\RockVariation.mid" \
    --prompt_file_path "C:\Users\Michael\Desktop\MusicDatasets\Datasets\RockDataset\processed\radiohead_2004-arpeggi.npy" \
    --number_generations 4 \
    --prompt_len 100 \
    --max_gen_len 120 \
    --min_temperature 0.75 \
    --max_temperature 1.1 \
    --min_top_p 0.85 \
    --max_top_p 0.98

C:\Users\Michael\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe: can't open file 'C:\\Users\\Michael\\Desktop\\Moonbeam-MIDI-Distillation\\recipes\\inference\\custom_music_generation\\generate_variations_midi.py': [Errno 2] No such file or directory
